# Explicit WFA Inversion (Time Series)

This notebook demonstrates the usage of the `neural_wfa` package for explicit (pixel-by-pixel) inversion on a simulated time series.
The time series is generated by amplifying the Stokes Q, U, V signals of a single snapshot.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as fits
import os
import sys

# Ensure src is in path if running locally
sys.path.append("src")
sys.path.append("../src")

from neural_wfa import Observation, WFAProblem
from neural_wfa.physics import LineInfo
from neural_wfa.optimization import PixelSolver
from neural_wfa.utils.viz import set_params
from neural_wfa.utils.viz import plot_wfa_results, torch2numpy

set_params()

## 1. Load Data & Simulate Time Series

In [ ]:
# Handle data path
datadir = "examples_py/plage_sst/"
if not os.path.exists(datadir):
    datadir = "plage_sst/"  # Fallback if running from inside dir

# Original Single Snapshot
# Shape: (Ny, Nx, 4, Nw)
img_raw = np.ascontiguousarray(
    fits.open(datadir + "CRISP_5173_plage_dat.fits", "readonly")[0].data,
    dtype="float32",
)
xl = np.ascontiguousarray(
    fits.open(datadir + "CRISP_5173_plage_wav.fits", "readonly")[0].data,
    dtype="float32",
)

print("Original Data shape:", img_raw.shape)

# --- Simulate Time Series ---
Nt = 4
amplify_factor = 1.5

# New Shape: (Nt, Ny, Nx, 4, Nw)
# effectively stacking frames
print(f"Generating {Nt} frames with Q/U/V amplification factor {amplify_factor}...")

img_series_list = []

for t in range(Nt):
    frame = img_raw.copy()

    # Scale factor for this timestep: 1.5^t
    # t=0: 1.0
    # t=1: 1.5
    # t=2: 2.25
    # t=3: 3.375
    scale = amplify_factor**t

    # Apply to Q, U, V (indices 1, 2, 3)
    # Stoke I (index 0) remains constant
    frame[:, :, 1, :] *= scale
    frame[:, :, 2, :] *= scale
    frame[:, :, 3, :] *= scale

    img_series_list.append(frame)

img_series = np.stack(img_series_list, axis=0)
print("Time Series Data shape:", img_series.shape)

## 2. Setup Problem

In [ ]:
# Force CPU for explicit examples to ensure stability (or CUDA if available and mem allows)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
print("Using device:", device)

# Observation
# Passing 5D array (Nt, Ny, Nx, 4, Nw)
# Observation class will detect Nt=4
obs = Observation(img_series, xl, active_wav_idx=[5, 6, 7], device=str(device))
print(f"Observation loaded. Nt={getattr(obs, 'nt', 1)}, Grid={obs.grid_shape}")

# Line Parameters
lin = LineInfo(5173)

# WFA Physics Engine
problem = WFAProblem(obs, lin, weights=[10, 10, 10], device=device)

## 3. Pixel Solver

In [ ]:
# Initialize Solver
# Note: PixelSolver automatically detects nt from problem/observation
solver = PixelSolver(problem, device=device)

# Initialize Parameters using Weak Field Approximation
print("Initializing parameters (Weak Field)...")
solver.initialize_parameters(method="weak_field")

# --- Check Initial Guess Dimensions ---
initial_field = solver.get_field()
print("Initial Field Shapes:")
print(f"  Blos Map: {initial_field.blos_map.shape} (Should be Nt, Ny, Nx)")
print(f"  Btrans Map: {initial_field.btrans_map.shape}")


# --- Plot Frame 0 vs Frame 3 Initial Guess ---
b_map = torch2numpy(initial_field.blos_map)
# Shape (Nt, Ny, Nx)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(b_map[0], origin="lower", cmap="gray")
plt.title("Frame 0: Blos (WFA Guess)")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(b_map[-1], origin="lower", cmap="gray")
plt.title(
    f"Frame {Nt - 1}: Blos (WFA Guess)\nScaled by {amplify_factor ** (Nt - 1):.2f}x roughly"
)
plt.colorbar()
plt.tight_layout()
plt.savefig("ref_time_series_init.png")
plt.close()
print("Saved initial comparison to ref_time_series_init.png")

# -------------------------------

print("Starting Inversion...")
solver.solve(
    n_iterations=50,  # Reduced iterations for example speed
    lr=1e-2,
    reguV=1e-4,
    reguQU=1e-2,
    # Temporal Regularization (Testing our new separation)
    reguT_Blos=1e-2,
    reguT_BQU=1e-2,
    # Magnitude Regularization (Testing newly added feature)
    reguMag_Blos=1e-4,
    reguMag_QU=1e-4,
    verbose=True,
)

## 4. Visualize Results & Analysis

In [ ]:
final_field = solver.get_field()
blos = torch2numpy(final_field.blos_map)
btrans = torch2numpy(final_field.btrans_map)
azi = torch2numpy(final_field.phi_map)

print("Final Result Shapes:", blos.shape)

# Plot last frame
plot_wfa_results(
    blos[-1], btrans[-1], azi[-1], save_name="ref_time_result_last_frame.png"
)
print("Saved result to ref_time_result_last_frame.png")